In [1]:
# Libraries
import pandas as pd
import numpy as np

In [2]:
# Load raw Stata file
df = pd.read_stata('/Users/maankumawat15/Downloads/Advance Stats Project Data/FINAL DATA-CITIZEN_SURVEY_ALL_DATA-FINAL_SKS_09.05.2025 (1)/FINAL DATA-CITIZEN_SURVEY_ALL_DATA-FINAL_SKS_09.05.2025.dta')
display(df.head())
df.describe()

,state_code,state_name,state_group,district_code,district_name,block_code,block_name,village_code,village_name,cs_id,...,d1m_1,d1m_2,d1m_3,rec_a2b_age,any_insurance,illness_op,illness_ip,uhc_index,uhc_index_terciles,_mergeuhcindex
0,Karnataka,Karnataka,C,Udupi,Udupi,5524,Karkal,803193,Karkal (TMC) WARD NO.-0009,23270,...,...,...,...,25-34,Have Insurance,NCD,Others,46.599998,Medium-UHC,matched (3)
1,Madhya Pradesh,Madhya Pradesh,B,Seoni,Seoni,3661,Ghansaur,1950,Deori,43617,...,no suggestions ...,...,...,25-34,No,Others,Pregnancy,49.459999,High-UHC,matched (3)
2,Kerala,Kerala,C,Thiruvananthapuram,Thiruvananthapuram,5689,Chirayinkeezhu,833,Koduvazhannoor,5341,...,...,...,...,15-24,Have Insurance,NCD,Others,50.349998,High-UHC,matched (3)
3,Madhya Pradesh,Madhya Pradesh,B,Shahdol,Shahdol,3691,Shahdol,2027,Harri,40619,...,No suggestions ...,...,...,25-34,No,Others,Pregnancy,45.439999,Medium-UHC,matched (3)
4,Mizoram,Mizoram,D,Serchhip,Serchhip,1910,Serchhip,803340,Serchhip (NT) WARD NO.-0004B,15701,...,...,...,...,25-34,Have Insurance,Others,Others,56.029999,High-UHC,matched (3)


,block_code,village_code,cs_id,consent,code_interviewer,code_supervisor,date_of_interview,a2a_mm,a2a_yy,a2b_age,...,rec_c11_per_month_annual,c11_per_year,c11_annual,c12,c14_per_month,rec_c14_per_month_annual,c14_per_year,c14_annual,c17_g_oth,uhc_index
count,50217.000000,5.021700e+04,50217.000000,50217.000000,50217.000000,50217.000000,5.021700e+04,50000.000000,50000.000000,50000.000000,...,2415.000000,806.000000,3221.000000,4.263000e+03,23217.000000,23217.000000,26654.000000,49871.000000,0.0,46301.000000
mean,3252.555927,4.132674e+05,25110.234084,1.004321,143.251489,206.870602,1.614068e+07,4.616000,1983.414820,38.585180,...,10202.351562,11762.107940,10592.654297,2.926233e+05,532.063402,6384.760742,3452.340812,4817.502930,NaN,45.004578
std,8848.256078,1.287388e+06,14498.417473,0.065595,109.287151,104.273759,8.905089e+06,3.154131,12.655289,12.655289,...,11227.237305,26522.671086,16456.566406,3.372824e+05,574.166069,6889.992676,14466.399795,11665.638672,NaN,7.114468
min,30.000000,1.000000e+00,1.000000,1.000000,1.000000,1.000000,1.012023e+06,1.000000,1950.000000,15.000000,...,1000.000000,500.000000,500.000000,0.000000e+00,100.000000,1200.000000,500.000000,500.000000,NaN,30.680000
25%,864.000000,6.950000e+02,12555.000000,1.000000,102.000000,201.000000,8.112022e+06,2.000000,1975.000000,29.000000,...,3600.000000,5000.000000,3600.000000,1.850000e+05,200.000000,2400.000000,1000.000000,1000.000000,NaN,40.599998
50%,2072.000000,1.947000e+03,25109.000000,1.000000,108.000000,201.000000,1.701202e+07,4.000000,1985.000000,37.000000,...,6000.000000,9000.000000,6000.000000,2.000000e+05,300.000000,3600.000000,1000.000000,2000.000000,NaN,43.320000
75%,3921.000000,8.031750e+05,37663.000000,1.000000,162.000000,205.000000,2.402202e+07,7.000000,1993.000000,47.000000,...,12000.000000,12000.000000,12000.000000,4.000000e+05,600.000000,7200.000000,2000.000000,6000.000000,NaN,49.459999
max,99998.000000,8.042729e+06,50227.000000,2.000000,997.000000,509.000000,3.112202e+07,12.000000,2007.000000,72.000000,...,81600.000000,500000.000000,500000.000000,5.000000e+06,4800.000000,57600.000000,500000.000000,500000.000000,NaN,66.669998


In [3]:
# Remove rows where consent was not given
df = df[df['consent'] != 2]

# Columns to drop
cols_to_drop = [
    # Admin/identifiers
    "_mergeuhcindex", "cs_id",
    "code_interviewer", "name_interviewer",
    "code_supervisor", "name_supervisor",
    "date_of_interview", "consent_adult",

    # Geography (keeping state_name, district_code, residence)
    "state_code", "district_name",
    "block_code", "village_code",
    "assembly_constituency_name", "parliamentary_constituency_name",

    # Demographic duplicates/free-text
    "a2a_mm", "a2a_yy",        # age already in a2b_age
    "a2d", "a2e",              # raw codes, recoded versions kept
    "a2e_oth", "a2h_oth", "a2i_oth",

    # OP free-text/others-specify
    "b1_all_oth", "b1_most_oth", "b3_oth", "b6g_oth",
    "b7_1_oth", "b7_2o_spo_oth",

    # OP multi-select checkboxes (b1_most captures most recent)
    *[f"b1_all_{x}" for x in [1,2,3,4,5,6,7,8,9,10,11,99,0,88]],

    # OP satisfaction/experience ratings
    "b6i","b6j","b6k","b6l","b6m","b6n","b6o","b6p",
    "b6q","b6r","b6s","b6t","b6u","b6v","b6w","b6x","b6y", "b6g",

    # IP free-text/others-specify
    "c1_all_oth", "c1_most_oth", "c3_oth", "c7d_oth",
    "c8_oth", "c9_o_spo_oth", "c16_i_spo_oth", "c17_g_oth", "c11",

    # IP multi-select checkboxes
    *[f"c1_all_{x}" for x in [1,2,3,4,5,6,99,0,88]],

    # IP satisfaction/experience ratings
    "c7d","c7e","c7f","c7g","c7h","c7i","c7j","c7k",
    "c7l","c7m","c7n","c7o","c7p","c7q","c7r","c7s","c9_o", "c1_all_oth",
    "c9_a","c9_b", "c9_c", "c9_d", "c9_e", "c9_f", "c9_g", "c9_h", "c9_i", "c9_j", "c9_k","c9_l", "c9_m", "c9_n",

    # Insurance — redundant premium columns (keeping c11_annual)
    "c11_per_month", "rec_c11_per_month_annual", "c11_per_year",
    "c10c", "c10f",

    # WTP hypothetical columns
    "c14", "c14_per_month", "rec_c14_per_month_annual", "c14_per_year", "c14_annual",

    # Jan-Aushadhi, health info sources, digital health, health card attitudes
    "c15",
    "c16_a","c16_b","c16_c","c16_d","c16_e","c16_f","c16_g","c16_h","c16_i",
    "c17_a","c17_b","c17_c","c17_d","c17_e","c17_f","c17_g",
    "c18_a", "c18_b",

    # Section D — preferences/opinions
    "d1a","d1b","d1c","d1d","d1e","d1f","d1g","d1h","d1i","d1j","d1k","d1l",
    "d1m_1","d1m_2","d1m_3",
]
df = df.drop(columns=cols_to_drop, errors='ignore')

# Drop OP provider preference reason columns (b7_2*)
df = df.drop(columns=[col for col in df.columns if col.startswith("b7_2")], errors='ignore')

print("Shape after column drops:", df.shape)

Shape after column drops: (50000, 65)


In [4]:
# Check visit distribution
print("Only OP visit:", (df['b1_most'].notna() & df['c1_most'].isna()).sum())
print("Only IP visit:", (df['b1_most'].isna() & df['c1_most'].notna()).sum())
print("Both visits:  ", (df['b1_most'].notna() & df['c1_most'].notna()).sum())
print("Neither visit:", (df['b1_most'].isna() & df['c1_most'].isna()).sum())

# Drop rows with neither OP nor IP visit — no expenditure data to analyse
df = df[~(df['b1_most'].isna() & df['c1_most'].isna())]
print("\nShape after dropping no-visit rows:", df.shape)

Only OP visit: 22811
Only IP visit: 964
Both visits:   19024
Neither visit: 7201

Shape after dropping no-visit rows: (42799, 65)


In [5]:
# Visit flags
df['had_op_visit'] = df['b1_most'].notna().astype(int)
df['had_ip_visit'] = df['c1_most'].notna().astype(int)
df['had_any_visit'] = ((df['had_op_visit'] == 1) | (df['had_ip_visit'] == 1)).astype(int)

# 1. Net OOP after insurance reimbursement
df['net_oope_op']    = df['oope_total_op'] - df['b6f'].fillna(0)
df['net_oope_ip']    = df['oope_total_ip'] - df['c7c'].fillna(0)
df['net_oope_total'] = df['net_oope_op'].fillna(0) + df['net_oope_ip'].fillna(0)

# 2. Household consumption proxy (food + education spend)
df['total_hh_spend'] = df['a2g_food'].fillna(0) + df['a2g_edu'].fillna(0)

# 3. OOP as share of household consumption
df['oope_share_income'] = np.where(
    df['total_hh_spend'] > 0,
    df['net_oope_total'] / df['total_hh_spend'],
    np.nan
)

# 4. Catastrophic expenditure flags (WHO thresholds)
df['catastrophic_10'] = (df['oope_share_income'] > 0.10).astype(int)
df['catastrophic_who_strict'] = np.where(
    (df['total_hh_spend'] - df['a2g_food'].fillna(0)) > 0,
    (df['net_oope_total'] / (df['total_hh_spend'] - df['a2g_food'].fillna(0)) > 0.40).astype(int),
    np.nan
)

# 5. Private facility flags — verify codes against codebook
PRIVATE_CODES = [3, 4, 5]
df['op_private'] = df['b1_most'].isin(PRIVATE_CODES).astype(int)
df['ip_private'] = df['c1_most'].isin(PRIVATE_CODES).astype(int)
df.loc[df['had_op_visit'] == 0, 'op_private'] = np.nan
df.loc[df['had_ip_visit'] == 0, 'ip_private'] = np.nan

# 6. Insurance effectiveness (reimbursed > 50% of bill)
df['insurance_effective'] = (
    (df['b6f'].fillna(0) / df['oope_total_op'].replace(0, np.nan) > 0.5) |
    (df['c7c'].fillna(0) / df['oope_total_ip'].replace(0, np.nan) > 0.5)
).astype(int)

# 7. Multi-episode burden
df['multi_episode'] = (df['had_op_visit'] + df['had_ip_visit'] > 1).astype(int)

# 8. Income quintiles (1=poorest, 5=richest)
df['income_quintile'] = pd.qcut(
    df['a2g_food'], q=5, labels=[1,2,3,4,5], duplicates='drop'
).astype(float)

# 9. Log-transformed total OOP (handles right skew)
df['log_net_oope_total'] = np.log1p(df['net_oope_total'])

# 10. Drop columns absorbed into engineered features
df = df.drop(columns=[
    'b6f', 'c7c',              # reimbursement → absorbed into net_oope
    'prop_indv_spent_oop_op',  # replaced by oope_share_income
    'prop_indv_spent_oop_ip',
    'the_op', 'the_ip',        # pre-reimbursement totals → replaced by net_oope
], errors='ignore')

#Removing empty entries in log_net_oope_total column
# errors='coerce' will turn any remaining non-numeric strings (like " ") into NaN
df['log_net_oope_total'] = pd.to_numeric(df['log_net_oope_total'].astype(str).str.strip(), errors='coerce')

# 2. Drop rows where the value is now NaN (optional, but recommended for regression)
df = df.dropna(subset=['log_net_oope_total'])

# 3. Double check the result
print(df['log_net_oope_total'].dtype)

print("Shape after feature engineering:", df.shape)

float64
Shape after feature engineering: (40070, 75)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [6]:
#Renaming columns for clarity and consistency:

df = df.rename(columns={
    # Geography & Identifiers
    'state_name'             : 'state',
    'state_group'            : 'state_region',
    'district_code'          : 'district',
    'block_name'             : 'block',
    'village_name'           : 'village',
    'residence'              : 'rural_urban',
    'cs_id'                  : 'survey_id',

    # Demographics
    'a2b_age'                : 'age',
    'rec_a2b_age'            : 'age_group',
    'a2c'                    : 'gender',
    'a2d'                    : 'education_raw',
    'rec_a2d'                : 'education',
    'a2e'                    : 'occupation_raw',
    'rec_a2e'                : 'occupation',
    'a2f'                    : 'household_size',
    'a2g_food'               : 'monthly_food_spend',
    'a2g_edu'                : 'monthly_edu_spend',
    'a2h'                    : 'religion',
    'a2i'                    : 'caste',
    'a2j'                    : 'marital_status',
    'a3'                     : 'physical_health_rating',
    'a4'                     : 'mental_health_rating',

    # OP (Out-Patient) Visit Details
    'b1_most'                : 'op_facility_type',
    'b2'                     : 'op_patient_identity',
    'b3'                     : 'op_reason',
    'b4'                     : 'op_is_emergency',
    'b5'                     : 'op_months_ago',
    'b6i'                    : 'op_travel_time',
    'b6m'                    : 'op_wait_time',
    'b7_1'                   : 'op_preferred_facility',

    # OP Expenditure (Corrected Mappings)
    'b6a'                    : 'op_cost_consultation',
    'b6b'                    : 'op_cost_is_reasonable',
    'b6c'                    : 'op_cost_medicine',
    'b6d'                    : 'op_used_ayush',
    'b6e'                    : 'op_cost_diagnostics',
    'medical_op'             : 'op_cost_medical_total',
    'b6f'                    : 'op_insurance_reimbursement',
    'b6h'                    : 'op_cost_transport',
    'the_op'                 : 'op_cost_total_combined', # Med + Non-Med
    'oope_medical_op'        : 'op_oope_medical',
    'oope_total_op'          : 'op_oope_total',

    # IP (In-Patient) Visit Details
    'c1_most'                : 'ip_facility_type',
    'c2'                     : 'ip_patient_identity',
    'c3'                     : 'ip_reason',
    'c4'                     : 'ip_months_ago',
    'c5'                     : 'ip_is_emergency',
    'c6'                     : 'ip_length_of_stay',
    'c8'                     : 'ip_preferred_facility',

    # IP Expenditure (Corrected Mappings)
    'c7a'                    : 'ip_cost_treatment_total',
    'medical_ip'             : 'ip_cost_medical_total',
    'c7c'                    : 'ip_insurance_reimbursement',
    'the_ip'                 : 'ip_cost_total_combined',
    'oope_medical_ip'        : 'ip_oope_medical',
    'oope_total_ip'          : 'ip_oope_total',
    'c7f'                    : 'ip_cost_transport',
    'c7g'                    : 'ip_travel_time',

    # Insurance Specifics
    'c10a'                   : 'ins_govt_ayushman',
    'c10b'                   : 'ins_state_scheme',
    'c10d'                   : 'ins_employer_scheme',
    'c10e'                   : 'ins_private',
    'c11'                    : 'ins_payment_mode',
    'c11_annual'             : 'ins_premium_annual',
    'c12'                    : 'ins_coverage_limit',
    'c13'                    : 'ins_adequacy_perception',
    'any_insurance'          : 'has_insurance',

    # Indices & Flags
    'uhc_index'              : 'district_uhc_index',
    'uhc_index_terciles'     : 'district_uhc_tercile',
    'illness_op'             : 'op_illness_category',
    'illness_ip'             : 'ip_illness_category'
})

print("Final shape:", df.shape)
print(df.columns.tolist())

Final shape: (40070, 75)
['state', 'state_region', 'district', 'block', 'village', 'consent', 'rural_urban', 'age', 'gender', 'education', 'occupation', 'household_size', 'monthly_food_spend', 'monthly_edu_spend', 'religion', 'caste', 'marital_status', 'physical_health_rating', 'mental_health_rating', 'op_facility_type', 'op_patient_identity', 'op_reason', 'op_is_emergency', 'op_months_ago', 'op_cost_consultation', 'op_cost_is_reasonable', 'op_cost_medicine', 'op_used_ayush', 'op_cost_diagnostics', 'op_cost_medical_total', 'op_cost_transport', 'op_oope_medical', 'op_oope_total', 'op_preferred_facility', 'ip_facility_type', 'ip_patient_identity', 'ip_reason', 'ip_months_ago', 'ip_is_emergency', 'ip_length_of_stay', 'ip_cost_treatment_total', 'ip_cost_medical_total', 'c7b', 'ip_oope_medical', 'ip_oope_total', 'ip_preferred_facility', 'ins_govt_ayushman', 'ins_state_scheme', 'ins_employer_scheme', 'ins_private', 'ins_premium_annual', 'ins_coverage_limit', 'ins_adequacy_perception', 'age_g

In [7]:
# 1. Define the categories that are Private
private_list = ["PrivateClinic", "PrivateHospital"]

# 2. Create indicators: 1 if Private, 0 if Public/Other
df['op_private'] = df['op_facility_type'].isin(private_list).astype(int)
df['ip_private'] = df['ip_facility_type'].isin(private_list).astype(int)

# 3. Quick verification
print("Mapping Verification:")
print(df.groupby('op_facility_type')['op_private'].value_counts())

Mapping Verification:
op_facility_type         op_private
ASHA                     0              1107
                         1                 0
Sub Centre               0              1429
                         1                 0
PHC                      0              5354
                         1                 0
CHC                      0              3177
                         1                 0
GovernmentHospital       0             10605
                         1                 0
PrivateClinic            1              9115
                         0                 0
PrivateHospital          1              2397
                         0                 0
Doctor-MobileVan         0               245
                         1                 0
AYUSH                    0               256
                         1                 0
TraditionalHealer/Quack  0              3735
                         1                 0
Chemist                  0              17

/var/folders/sq/j91jgq3545qb3_p0dg936cp80000gn/T/ipykernel_10231/1691144756.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('op_facility_type')['op_private'].value_counts())


In [8]:
# Save preprocessed dataset
df.to_csv("preprocessed_data.csv", index=False)
print("Saved to preprocessed_data.csv")

Saved to preprocessed_data.csv
